# Use case — `Utility/time_delay_pspline.py`

Adaptive shared P-spline estimation for two, three or four components. The first component is the zero-delay reference.

**Convention:** `t_shifted_k = t_k - delay_k`. Inspect LOO, overlap and K profiles before interpreting the selected delay.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import time_delay_pspline as td

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # Canonical preprocessed light curves.
SOURCE_ID = "5915407711751697280"                               # System containing all requested components.
COMPONENT_IDS = [                                                # Two to four IDs; first entry defines delay zero.
    "5915407711751697345",
    "5915407711751697347",
]
COMPONENT_NAMES = ["A reference", "B"]                         # Human-readable labels matching COMPONENT_IDS.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

## Estimator hyperparameters

This complete dictionary preserves the original `TDPSPL.ipynb` adaptation. For a first run, replace it with `td.load_parameter_profile("quick")`.

In [ ]:
estimator_kwargs = {
    "dmin": -500,                         # Minimum delay for every non-reference component, in days.
    "dmax": 500,                          # Maximum delay for every non-reference component, in days.
    "ngrid": 400,                         # Coarse delay-grid values evaluated per coordinate.
    "degree": 3,                          # Cubic B-spline degree.
    "max_delay_passes": 1,                # Coordinate-search passes over all free delays.
    "points_per_interval": 999,           # Legacy density rule; LOO K selection dominates with this high value.
    "K_min": 5,                           # Smallest number of internal quantile knots considered.
    "K_max": 150,                         # Largest number of internal knots allowed by data and LOO.
    "K_selection": "loo",                 # Select spline complexity using fast leave-one-out error.
    "K_loo_candidates": None,             # None builds a data-adaptive candidate grid; list fixes exact candidates.
    "K_loo_n_candidates": 15,             # Number of K values in the adaptive LOO candidate grid.
    "K_loo_complexity_penalty": 1.0,      # Additional LOO penalty discouraging unnecessarily large K.
    "K_loo_df_cap_frac": 0.80,            # Reject K when effective degrees of freedom exceed this data fraction.
    "delay_score": "loo",                 # Delay objective; retained alternatives include BIC-like scoring.
    "rolling_window": 3,                  # Local points used for the global rolling median/MAD noise estimate.
    "rolling_mode": "points",             # "points" uses neighbour count; "time" uses a day radius.
    "rolling_time_radius": None,          # Radius in days for rolling_mode="time"; None estimates it from cadence.
    "use_global_mad_in_sigma": True,      # Combine rolling intrinsic variability with measurement sigma.
    "lambda_base": "auto",                # Base P-spline roughness penalty; "auto" derives it from global MAD.
    "lambda_scale": 0.03,                 # Multiplicative scale applied to the automatically derived lambda.
    "lambda_alpha": 0.5,                  # Strength of local lambda adaptation to the rolling MAD.
    "lambda_min_ratio": 0.05,             # Minimum local lambda divided by base lambda.
    "lambda_max_ratio": 5.0,              # Maximum local lambda divided by base lambda.
    "mad_floor": 0.05,                    # Minimum normalised robust dispersion and measurement sigma.
    "df_cap_frac": 0.30,                  # Effective-DF cap used by the BIC-like diagnostic.
    "rough_penalty": 1.0,                 # Weight of spline curvature in the BIC-like diagnostic.
    "overlap_penalty": 0.0,               # Soft cost for reduced overlap beyond the hard validity rules.
    "min_span_frac_ref": 1.0,             # Reference span used to scale the optional overlap penalty.
    "min_frac_ref": 1.0,                  # Reference point fraction used to scale the optional overlap penalty.
    "min_points": 10,                     # Minimum shifted-overlap points required per component.
    "min_frac": 0.50,                     # Minimum fraction of each component retained after shifting.
    "min_span_frac": 0.50,                # Minimum common temporal span relative to the original span.
    "scalar_xatol": 0.05,                 # Bounded local-refinement tolerance in days.
    "force_final_to_min_plotted_loo": True, # Align the final reported solution with the plotted LOO minimum.
    "verbose": True,                      # Print coordinate-search progress and final diagnostics.
}

MC_SAMPLES = 300                 # Flux-error draws; use 20 for a smoke test and >=300 for final work.
MC_RANDOM_SEED = 42              # Reproducible random-number seed.
MC_ERROR_SCALE = 1.0             # 1.0 uses flux_obs_error exactly.
MC_FORCE_SAME_K = True           # Reuse base-fit K; False propagates K selection but is much slower.
MC_PROGRESS_EVERY = 10           # Print progress every N draws.

In [ ]:
df = td.load_lightcurve_csv(INPUT_CSV)
system = td.get_components_from_df(
    df=df,
    source_id=SOURCE_ID,
    comp_ids=COMPONENT_IDS,
    names=COMPONENT_NAMES,
)
result = td.estimate_time_delay_pspline(
    system["curves"],
    **estimator_kwargs,
)
td.print_result_multi(system, result)
display(result["pair_delays"])

## Required diagnostics

A delay is not accepted from its scalar value alone. Inspect the objective, LOO, overlap, selected K, fitted curves, rolling-MAD sigma and residuals.

In [ ]:
td.plot_delay_profiles_multi(result, y_col="cost")
td.plot_LOO_score_profile(result)
td.plot_overlap_profile(result)
td.plot_K_profile(result)
td.plot_K_loo_table(result)
td.plot_fit_multi(result, degree=estimator_kwargs["degree"], show_knots=True)
td.plot_global_mad_sigma_diagnostics(result)
td.plot_residuals_multi(result)

## Measurement-error uncertainty

Every draw perturbs each flux by its own `flux_obs_error`, then reruns the estimator. The sample can be multimodal; always inspect its histogram and saved draws.

In [ ]:
uncertainty = td.run_fluxobs_error_mc_pspline(
    system=system,
    estimator_kwargs=estimator_kwargs,
    n_samples=MC_SAMPLES,
    random_seed=MC_RANDOM_SEED,
    error_scale=MC_ERROR_SCALE,
    base_res=result,
    force_same_K_as_base=MC_FORCE_SAME_K,
    progress_every=MC_PROGRESS_EVERY,
    verbose=True,
)
td.print_fluxobs_error_mcmc_uncertainty(uncertainty)
td.plot_fluxobs_error_mcmc_uncertainty(uncertainty, bins=30)
display(uncertainty["pair_delay_summary"])

## Batch command

```bash
python -m Utility.time_delay_pspline data/cleaned_lightcurves.csv --pairs configs/time_delay_system_pairs.csv --profile legacy_notebook --mc-samples 300 --output-dir results/batch_time_delays
```